# Video AI Landscape & Data Needs

A data-first orientation to the video generation field. Before diving into architectures or training code, understand *what data each product needs* and *why the datasets team is upstream of everything*.

**Time:** ~1–1.5 hours (reading + reflection) | **Code:** Minimal — this is a conceptual orientation notebook.

## Product Evolution: Text → Image → Video → World Models

The generative AI stack is evolving through distinct capability tiers, each with escalating data requirements:

| Generation | Capability | Example Products | Key Data Need |
|---|---|---|---|
| **Gen 1** | Text → Image | Image generators | Large-scale image-caption pairs, aesthetic filtering |
| **Gen 2** | Text → Video | Short-form video clips (2-10s) | Temporally dense captions, motion quality filtering |
| **Gen 3** | Image/Video → Video | Video editing, style transfer, in-painting | Paired before/after data, instruction-following |
| **Gen 4** | Interactive generation | Real-time rendering, controllable cameras | 3D-consistent scene data, action-conditioned sequences |
| **Gen 5** | World models | Physics simulation, embodied AI, game engines | Robot command→video pairs, physics-aware annotations |

Each generation doesn't replace the previous — it builds on top of it. A world model lab still needs Gen 1-3 data pipelines running at scale.

### The capability flywheel

```mermaid
graph LR
    A[Better Data] --> B[Better Models]
    B --> C[Better Products]
    C --> D[More Users / Revenue]
    D --> E[More Compute Budget]
    E --> A
```

Data quality is the entry point of this flywheel. The datasets team directly controls the first lever.

## Per-Product Data Requirements

### Text-to-Video
- **Input data:** Video clips + temporally dense captions (not just a single sentence — describe what happens at each moment)
- **Volume:** 100M–1B+ video-caption pairs
- **Quality bar:** Motion coherence, no watermarks, no static slideshows, resolution ≥ 720p
- **Caption quality:** Structured format: scene description, motion description, camera movement, style tags
- **Key challenge:** Most web-scraped video has garbage alt-text. Re-captioning with VLMs is mandatory.

### Video Editing (in-context)
- **Input data:** Paired before/after editing examples with edit instructions
- **Volume:** 10M–100M instruction-edit pairs (much harder to source)
- **Quality bar:** Edit must be visible, instruction must be accurate, no hallucinated edits
- **Key challenge:** Synthetic data generation — real paired data is scarce, so you generate it programmatically

### Action-Conditioned World Models
- **Input data:** Action (joystick command, robot arm pose, camera trajectory) → resulting video sequence
- **Volume:** Millions of action-conditioned trajectories
- **Quality bar:** Precise temporal alignment between action and visual outcome
- **Key challenge:** Sim-to-real gap — simulator data is cheap but doesn't transfer perfectly

### Audio-Driven Avatars
- **Input data:** Tightly aligned audio-visual data (speech ↔ facial motion ↔ gesture)
- **Volume:** 10K–100K hours of talking-head video with aligned audio
- **Quality bar:** Lip sync precision, natural gesture timing, no identity leakage across subjects
- **Key challenge:** Temporal alignment at millisecond granularity

### Interactive Real-Time Generation
- **Input data:** 3D-consistent multi-view scene data, user interaction logs
- **Volume:** Varies widely — 3D data is expensive to create
- **Quality bar:** Multi-view consistency, physics plausibility
- **Key challenge:** 3D ground truth is orders of magnitude more expensive than 2D video

## The Data Team as Upstream Enabler

### Why "transversal"

Every product line at a video AI lab flows through the same datasets infrastructure:

```
Raw video sources
    ↓
[Ingestion & crawling]
    ↓
[Deduplication]         ← Shared infrastructure
    ↓
[Quality filtering]     ← Shared infrastructure
    ↓
[Scene segmentation]    ← Shared infrastructure
    ↓
[Re-captioning (VLM)]   ← Shared infrastructure
    ↓
┌──────────┬──────────┬──────────┐
│ T2V data │ Edit data│ WM data  │  ← Product-specific branches
│ mixing   │ pairing  │ action   │
│ & shards │ & shards │ & shards │
└──────────┴──────────┴──────────┘
    ↓           ↓          ↓
[Training cluster A] [Cluster B] [Cluster C]
```

The upstream stages (ingestion through re-captioning) are shared. The downstream stages (data mixing, format-specific sharding) are product-specific. This is why the datasets team is *transversal* — it serves every research and product team.

### What the data team owns
- **Data ingestion:** Crawling, licensing, partnership data onboarding
- **Data quality:** Dedup, filtering, annotation quality
- **Data transformation:** Re-captioning, format conversion, shard packing
- **Data serving:** WebDataset/MDS shards served to distributed training
- **Data tooling:** Internal tools for data exploration, debugging, monitoring
- **Data metrics:** Coverage, diversity, quality scores, training signal strength

## Data Quality Under Aggressive Compression

### Why data quality matters *more* in modern architectures

Modern video diffusion models use aggressive latent space compression. Stable Cascade's Würstchen architecture achieves **42× spatial compression** — a 1024×1024 image becomes a 24×24 latent.

**Implication:** Every latent token carries enormous semantic weight. A bad caption or low-quality frame doesn't just waste one data point — it corrupts a disproportionate amount of the model's learned representation.

| Compression Ratio | Latent Resolution (1024px input) | Semantic Density | Data Quality Sensitivity |
|---|---|---|---|
| 8× (SD 1.5) | 128 × 128 | Low | Moderate — model can learn around noise |
| 16× (SDXL) | 64 × 64 | Medium | Higher — quality filtering matters |
| 42× (Stable Cascade) | 24 × 24 | Very high | Critical — every bit counts |

### The data quality multiplier

At 42× compression:
- **Bad captions** teach the model wrong associations that are hard to unlearn
- **Duplicate data** wastes compute and biases the distribution (the model memorizes instead of generalizing)
- **Low-quality frames** (blur, watermarks, letterboxing) encode artifacts into the latent space
- **Temporal inconsistency** in video (jump cuts labeled as continuous) confuses motion learning

This is why the data pipeline notebooks (02–04) focus so heavily on deduplication, quality filtering, and re-captioning. These aren't nice-to-haves — they're the #1 lever for model quality.

## AR vs. Diffusion: Data Perspective

The field is converging on hybrid architectures that combine autoregressive (AR) and diffusion approaches. From a data perspective, they have different requirements:

### Diffusion models
- Can work with **unordered frames** — the model denoises all frames simultaneously
- Caption describes the *whole clip* — no need for per-frame temporal ordering in the caption
- Data augmentation is straightforward (random crop, flip, color jitter)
- **Data pipeline implication:** Simpler temporal requirements, but quality filtering is critical

### Autoregressive models
- Need **temporally causal** data — frame order matters because the model generates left-to-right
- Per-frame or per-segment captions are more valuable than whole-clip captions
- Data augmentation must preserve temporal ordering (no frame shuffling)
- **Data pipeline implication:** Temporal metadata (timestamps, scene boundaries) becomes essential

### Hybrid approaches (e.g., AR planning + diffusion rendering)
- Need data that supports *both* paradigms: ordered sequences with whole-clip and per-segment captions
- The AR component plans keyframes, the diffusion component fills in between
- **Data pipeline implication:** Hierarchical caption structure — clip-level + segment-level + frame-level annotations

### What this means for the datasets team

The shift from pure diffusion to hybrid AR+diffusion means:
1. **Temporal metadata** goes from optional to mandatory
2. **Caption granularity** increases (clip → segment → frame)
3. **Scene boundary detection** becomes a core pipeline stage
4. **Data formats** must support efficient sequential access (not just random access)

This directly shapes the tools you'll build in notebooks 02–04.

## Self-Assessment Q&A

Test your understanding. Click to reveal each answer.

---

<details>
<summary><strong>Q1: What kind of data does an action-conditioned world model need that a text-to-video model doesn't?</strong></summary>

Action-conditioned world models need **paired action→video sequences** where each frame is annotated with the action (joystick input, robot command, camera trajectory) that caused it. Text-to-video only needs text descriptions of the visual content. The world model must learn *causal relationships* between actions and visual outcomes, not just visual appearance.
</details>

---

<details>
<summary><strong>Q2: Why do temporally dense captions matter more than single-sentence descriptions?</strong></summary>

A single sentence like "a dog running in a park" doesn't capture *when* things happen — the dog starts walking, then runs, then stops to sniff. Temporally dense captions ("0-2s: golden retriever walks along path, 2-4s: breaks into a run across grass, 4-5s: stops and sniffs a bush") give the model much stronger learning signal for motion dynamics and temporal coherence.
</details>

---

<details>
<summary><strong>Q3: How does moving from diffusion-only to AR+diffusion change data requirements?</strong></summary>

Three key changes: (1) Frame ordering becomes mandatory — AR models generate causally, so data must preserve temporal order. (2) Caption granularity increases — you need per-segment or per-frame descriptions, not just clip-level. (3) Data formats must support efficient sequential access for AR training, not just random frame sampling.
</details>

---

<details>
<summary><strong>Q4: Why is re-captioning considered the highest-ROI data intervention?</strong></summary>

Web-scraped video comes with low-quality alt-text or no captions at all. Replacing these with structured VLM-generated captions (describing scene, motion, camera, style) dramatically improves text-video alignment during training. Studies consistently show this improves model quality more than architecture changes, more training compute, or larger datasets. It's high ROI because VLM inference is cheap relative to the quality improvement.
</details>

---

<details>
<summary><strong>Q5: What makes 3D/multi-view data so much more expensive than 2D video?</strong></summary>

2D video is abundant on the web (YouTube alone has billions of hours). 3D-consistent multi-view data requires either: (a) multi-camera capture rigs (expensive hardware + controlled environments), (b) 3D scanning/reconstruction (compute-intensive, often noisy), or (c) synthetic rendering (requires 3D assets and simulators). Each method costs 10-1000× more per data point than scraping 2D video.
</details>

---

<details>
<summary><strong>Q6: A video diffusion model produces blurry outputs. Before touching the model, what data issues would you investigate?</strong></summary>

(1) Resolution distribution — are low-res videos polluting the dataset? (2) Duplicate data — is the model memorizing a biased subset? (3) Motion quality — are there static/slideshow clips that teach the model to generate no motion? (4) Compression artifacts — are source videos heavily compressed (low bitrate)? (5) Caption-video misalignment — are captions describing different content than what's shown?
</details>

---

<details>
<summary><strong>Q7: Why does aggressive latent compression (42×) make data quality more important?</strong></summary>

At 42× compression, each latent token represents a large spatial region. Artifacts, bad captions, or quality issues in the input get "amplified" because they corrupt tokens that carry more semantic weight. At 8× compression the model has more tokens to work with and can learn around noise; at 42× there's no slack — garbage in → garbage out with higher fidelity.
</details>

---

<details>
<summary><strong>Q8: You're building a data pipeline for a new product: text-to-4D (generating 3D scenes that evolve over time). What stages would you add to the standard video pipeline?</strong></summary>

Beyond the standard video pipeline (ingest → dedup → quality filter → scene segment → re-caption → shard), you'd add: (1) **Multi-view consistency validation** — ensure different viewpoints of the same scene are geometrically consistent. (2) **Depth/geometry extraction** — monocular depth estimation or structure-from-motion to provide 3D supervision. (3) **Temporal 3D tracking** — track how 3D structures change over time, not just 2D optical flow. (4) **Physics plausibility scoring** — filter out sequences where objects move in physically impossible ways.
</details>

---

<details>
<summary><strong>Q9: What's the difference between data mixing and data curriculum?</strong></summary>

**Data mixing** is about *proportions* — what fraction of each data source to include in training (e.g., 40% high-quality video, 30% image-caption, 20% synthetic, 10% instruction-edit pairs). **Data curriculum** is about *ordering* — which data to show the model first (e.g., images first, then short videos, then long videos). Both are levers the datasets team controls. Mixing affects the final distribution; curriculum affects learning dynamics and convergence speed.
</details>

## What's Next

You now have a mental model of:
- The product evolution driving data requirements
- Per-product data needs and their challenges
- Why the datasets team is transversal (upstream of every product)
- How latent compression amplifies data quality impact
- How the AR↔diffusion shift changes data pipeline requirements

**Next notebook:** [02 — Video Data Pipeline](02_video_data_pipeline.ipynb) — hands-on pipeline: scene detection, filtering, captioning, embedding, and packing with Lance.